# HCR Activity Replay QA

Recall the read-only HCR-centric replay audit outputs for interpretation and QA. This notebook loads persisted manifests, accepted control tables, historical recompute artifacts, and label-image evidence; reusable replay logic stays in `src/codeants_2pf_hcr/`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from codeants_2pf_hcr import SingleFishPipelineConfig, resolve_pipeline_paths, stage_manifest_path

FISH_ID = "L395_f11"
LOCAL_ROOT = Path(os.environ.get("CODEANTS_LOCAL_ROOT", "/Volumes/dataDrive/dataProcessing/2p_processing"))
PIPELINE_ROOT = Path(os.environ.get("CODEANTS_HCR_REPLAY_PIPELINE_ROOT", "/tmp/codeants-hcr-replay-abzbAl/staged-L395"))
EXPLICIT_MANIFEST_PATH = os.environ.get("CODEANTS_HCR_REPLAY_MANIFEST", "")

ACCEPTED_REG_DIR = LOCAL_ROOT / FISH_ID / "03_analysis" / "functional" / "registration"
HISTORICAL_AUDIT_DIR = ACCEPTED_REG_DIR / "pipeline_outputs" / "assign-hcr-identity" / "recompute-audit"
config = SingleFishPipelineConfig(fish_id=FISH_ID, local_root=LOCAL_ROOT, strict=True, pipeline_root=PIPELINE_ROOT)
paths = resolve_pipeline_paths(config)

display(pd.DataFrame([
    {"name": "repo_root", "value": str(REPO_ROOT)},
    {"name": "fish_id", "value": FISH_ID},
    {"name": "local_root", "value": str(LOCAL_ROOT)},
    {"name": "pipeline_root", "value": str(PIPELINE_ROOT)},
    {"name": "accepted_reg_dir", "value": str(ACCEPTED_REG_DIR)},
    {"name": "historical_audit_dir", "value": str(HISTORICAL_AUDIT_DIR)},
]))

## Optional Audit Refresh

Set `RUN_AUDIT = True` to refresh the read-only audit manifest from the configured roots. Leave it `False` when you only want to recall existing outputs.

In [ ]:
RUN_AUDIT = False

audit_cmd = [
    "python",
    "tools/single_fish_pipeline.py",
    "audit-hcr-activity-replay",
    "--fish-id", FISH_ID,
    "--local-root", str(LOCAL_ROOT),
    "--strict",
    "--pipeline-root", str(PIPELINE_ROOT),
    "--write-manifest",
]
display(Markdown("`" + " ".join(audit_cmd) + "`"))

if RUN_AUDIT:
    completed = subprocess.run(audit_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout[-4000:])
    if completed.stderr:
        print(completed.stderr[-4000:])
    print("returncode", completed.returncode)

## Manifest Summary

In [ ]:
manifest_candidates = [
    Path(EXPLICIT_MANIFEST_PATH) if EXPLICIT_MANIFEST_PATH else None,
    stage_manifest_path(paths, "audit-hcr-activity-replay"),
    Path("/tmp/codeants-hcr-replay-abzbAl/audit_hcr_replay_variant_scoreboard_scaled_affine.json"),
]
manifest_path = next((p for p in manifest_candidates if p is not None and p.exists()), None)
manifest = {}
if manifest_path is None:
    display(Markdown("No manifest found. Run the audit cell or set `CODEANTS_HCR_REPLAY_MANIFEST`."))
else:
    with manifest_path.open() as f:
        manifest = json.load(f)
    display(Markdown(f"Loaded manifest: `{manifest_path}`"))

parameters = manifest.get("parameters", {})
summary_rows = [
    {"field": "stage_name", "value": manifest.get("stage_name", "")},
    {"field": "status", "value": manifest.get("status", "")},
    {"field": "errors", "value": len(manifest.get("errors", []))},
    {"field": "warnings", "value": len(manifest.get("warnings", []))},
    {"field": "promotion_enabled", "value": parameters.get("promotion_enabled", "")},
    {"field": "source_policy", "value": parameters.get("source_policy", "")},
    {"field": "plane_refs_summary_path", "value": parameters.get("plane_refs_summary_path", "")},
    {"field": "hcr_anatomy_root", "value": parameters.get("hcr_anatomy_root", "")},
]
display(pd.DataFrame(summary_rows))

checks_df = pd.DataFrame(manifest.get("checks", []))
if not checks_df.empty:
    display(checks_df[[c for c in ["status", "label", "message"] if c in checks_df.columns]])

## Replay Variant Scoreboard

In [ ]:
variant_df = pd.json_normalize(parameters.get("replay_variant_summaries", []))
if variant_df.empty:
    display(Markdown("No replay variant summaries found in the manifest."))
else:
    preferred_columns = [
        "status",
        "name",
        "row_delta_total",
        "counts.hcr_func_candidates.csv",
        "counts.conf_to_func_pairs_raw.csv",
        "counts.conf_to_func_pairs.csv",
        "candidate_key_gap.replay_keys",
        "candidate_key_gap.control_keys",
        "candidate_key_gap.missing_control_keys",
        "candidate_key_gap.extra_replay_keys",
        "plane_ref_report.backend_counts.ants_rigid_affine",
        "plane_ref_report.missing_transform_files",
    ]
    display(variant_df[[c for c in preferred_columns if c in variant_df.columns]].sort_values(["row_delta_total", "name"], na_position="last"))

## Candidate Tables

In [ ]:
table_paths = {
    "accepted_hcr_func_candidates": ACCEPTED_REG_DIR / "hcr_func_candidates.csv",
    "accepted_conf_to_func_pairs_raw": ACCEPTED_REG_DIR / "conf_to_func_pairs_raw.csv",
    "accepted_conf_to_func_pairs": ACCEPTED_REG_DIR / "conf_to_func_pairs.csv",
    "historical_recomputed_hcr_func_candidates": HISTORICAL_AUDIT_DIR / "hcr_func_candidates_recomputed.csv",
    "historical_variant_scoreboard": HISTORICAL_AUDIT_DIR / "hcr_transform_replay_variants.csv",
    "staged_assign_hcr_func_candidates": PIPELINE_ROOT / "assign-hcr-identity" / "registration" / "hcr_func_candidates.csv",
}
tables = {}
for name, path in table_paths.items():
    if path.exists():
        tables[name] = pd.read_csv(path)

display(pd.DataFrame([
    {"table": name, "exists": path.exists(), "rows": len(tables[name]) if name in tables else np.nan, "path": str(path)}
    for name, path in table_paths.items()
]))

if "accepted_hcr_func_candidates" in tables:
    display(tables["accepted_hcr_func_candidates"].head())
if "historical_variant_scoreboard" in tables:
    display(tables["historical_variant_scoreboard"].sort_values(["missing_control_keys", "extra_replay_keys"], na_position="last").head(20))

## Candidate Key Gap

In [ ]:
key_columns = ["gene", "anat_label", "plane_idx", "func_label"]
left_name = "accepted_hcr_func_candidates"
right_name = "historical_recomputed_hcr_func_candidates"

if left_name not in tables or right_name not in tables:
    display(Markdown("Accepted and historical recomputed candidate tables are both required for key-gap drill-down."))
elif any(c not in tables[left_name].columns for c in key_columns) or any(c not in tables[right_name].columns for c in key_columns):
    display(Markdown("One of the candidate tables is missing expected key columns."))
else:
    accepted_keys = tables[left_name][key_columns].fillna("").astype(str).drop_duplicates().copy()
    recomputed_keys = tables[right_name][key_columns].fillna("").astype(str).drop_duplicates().copy()
    accepted_keys["candidate_key"] = accepted_keys[key_columns].agg("||".join, axis=1)
    recomputed_keys["candidate_key"] = recomputed_keys[key_columns].agg("||".join, axis=1)
    accepted_key_set = set(accepted_keys["candidate_key"])
    recomputed_key_set = set(recomputed_keys["candidate_key"])
    missing_from_recomputed = accepted_keys[accepted_keys["candidate_key"].isin(sorted(accepted_key_set - recomputed_key_set))]
    extra_in_recomputed = recomputed_keys[recomputed_keys["candidate_key"].isin(sorted(recomputed_key_set - accepted_key_set))]
    display(pd.DataFrame([
        {"comparison": "accepted keys", "n": len(accepted_key_set)},
        {"comparison": "historical recomputed keys", "n": len(recomputed_key_set)},
        {"comparison": "accepted missing from historical recompute", "n": len(missing_from_recomputed)},
        {"comparison": "historical recompute extras", "n": len(extra_in_recomputed)},
    ]))
    display(Markdown("Missing accepted keys:"))
    display(missing_from_recomputed.head(50))
    display(Markdown("Extra historical recompute keys:"))
    display(extra_in_recomputed.head(50))

## Historical Warped Functional Labels

In [ ]:
label_paths = sorted(HISTORICAL_AUDIT_DIR.glob("best_replay_recomputed_func_labels_plane*.tif"))
display(pd.DataFrame([{"plane_image": p.name, "path": str(p)} for p in label_paths]))

if not label_paths:
    display(Markdown("No historical warped label TIFFs found."))
else:
    import tifffile
    n_images = len(label_paths)
    fig, axes = plt.subplots(1, n_images, figsize=(4 * n_images, 4), squeeze=False)
    image_stats = []
    for ax, path in zip(axes.ravel(), label_paths):
        arr = tifffile.imread(path)
        view = arr.max(axis=0) if arr.ndim > 2 else arr
        labels = np.unique(arr)
        labels = labels[labels != 0]
        image_stats.append({"image": path.name, "shape": tuple(arr.shape), "nonzero_px": int(np.count_nonzero(arr)), "label_count": int(len(labels))})
        ax.imshow(view > 0, cmap="gray")
        ax.set_title(path.stem.replace("best_replay_recomputed_func_labels_", ""))
        ax.axis("off")
    plt.show()
    display(pd.DataFrame(image_stats))

## Current Interpretation Notes

Use this cell for short manual QA notes after reviewing the manifest, candidate gaps, and warped-label evidence.